# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aneeqahabib/FlyRank_ML_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Contract answer:** One analysis row is one pseudonymized `client_hash_id` × `content_hash_id` pair aggregated over the March 2026 snapshot month. I use the warehouse table `fact_content_daily_performance`; its raw grain is daily `report_date` × client × content. Features are measurements available through 2026-03-31. The forward label is `is_declining_next_month`: 1 when April 2026 impressions are below 80% of March impressions, and 0 otherwise. This supports a directional content-review queue; it does not claim to predict or cause search rankings.

**Deliberate exclusion:** I exclude April performance from the feature vector because it is only known after the March 31 decision moment.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip -q install duckdb huggingface_hub

import os
import duckdb
import pandas as pd

# Colab: store HF_TOKEN in the Secrets panel and enable notebook access.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "Create a Colab Secret named HF_TOKEN, enable notebook access, and rerun this cell."
    )

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet', hive_partitioning=true)"
MONTH = "2026-03"

print("Connected to the warehouse")
print("Contract snapshot month:", MONTH)
print("Raw source grain: report_date x client x content")


Connected to the warehouse
Contract snapshot month: 2026-03
Raw source grain: report_date x client x content


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature fields, all knowable at the 2026-03-31 decision moment:** `impressions_march`, `clicks_march`, `avg_position_march`, `sessions_march`, and `engagement_rate_march` (engaged sessions divided by sessions, when GA4 is available).

**Label / proxy:** `is_declining_next_month`, defined after March as April impressions `< 0.8 ×` March impressions. It is an observed forward outcome, not a causal claim.

**Context:** `client_hash_id`, `content_hash_id`, `report_date`, the March month key, and availability flags. These support grouping, joins, filtering, and auditing; IDs are never model inputs.

**Deliberately excluded:** April performance columns and any April-derived rate because they are future information at the decision moment; `gsc_avg_position` from April for the same reason; and `trend_direction` / `trend_pct` because they directly define a decline label in the starter data. Client and content IDs are also excluded from the model because pseudonyms can memorize entities rather than generalize.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Decision moment: 2026-03-31")
print("Features: March measurements only")
print("Label: April impressions below 80% of March impressions")
print("Excluded: future April fields, IDs, and label-derived trend fields")

Decision moment: 2026-03-31
Features: March measurements only
Label: April impressions below 80% of March impressions
Excluded: future April fields, IDs, and label-derived trend fields


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The first checks the documented raw fact grain. The second checks the March slice's size and date span. The third checks GA4 availability using `IS TRUE`, because the flag is three-valued.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Verification query 1: raw fact grain. Empty output means no duplicate daily keys.
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS duplicate_rows
    FROM {FACT}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Query 1 - duplicate raw daily keys (expected: 0 rows):")
display(grain_check)

# Verification query 2: March row count and date span.
count_window = con.sql(f"""
    SELECT COUNT(*) AS march_rows,
           COUNT(DISTINCT client_hash_id) AS clients,
           COUNT(DISTINCT content_hash_id) AS contents,
           MIN(report_date) AS first_date,
           MAX(report_date) AS last_date
    FROM {FACT}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()
print("Query 2 - March slice count and date span:")
display(count_window)

# Verification query 3: availability, explicitly treating NULL as not available.
availability = con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_unavailable_or_unknown_rows
    FROM {FACT}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()
print("Query 3 - GA4 availability using IS TRUE:")
display(availability)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 - duplicate raw daily keys (expected: 0 rows):


,report_date,client_hash_id,content_hash_id,duplicate_rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 2 - March slice count and date span:


,march_rows,clients,contents,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 3 - GA4 availability using IS TRUE:


,march_rows,ga4_available_rows,ga4_unavailable_or_unknown_rows
0,9841378,413966,9427412


## 3a. Five-feature frame and availability notes

*Build the smallest useful March frame. Each feature is available when the March snapshot closes.*

1. `impressions_march` — available on 2026-03-31 because it sums observed GSC impressions through that date.
2. `clicks_march` — available on 2026-03-31 because it sums observed GSC clicks through that date.
3. `avg_position_march` — available on 2026-03-31 because it averages observed daily GSC position through that date.
4. `sessions_march` — available on 2026-03-31 only for rows where GA4 availability is true.
5. `engagement_rate_march` — available on 2026-03-31, calculated from March engaged sessions and sessions, with missing/zero sessions kept as missing rather than treated as zero engagement.

The feature query also creates the forward April label for evaluation. April values are retained only as the observed outcome and are never included in the feature list.

In [4]:
# Aggregate March features and the April forward outcome inside DuckDB.
feature_frame = con.sql(f"""
    WITH monthly AS (
        SELECT
            client_hash_id,
            content_hash_id,
            DATE_TRUNC('month', report_date) AS month_start,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_position,
            SUM(ga4_sessions) FILTER (WHERE ga4_data_available IS TRUE) AS sessions,
            SUM(ga4_engaged_sessions) FILTER (WHERE ga4_data_available IS TRUE) AS engaged_sessions,
            BOOL_OR(ga4_data_available IS TRUE) AS ga4_available
        FROM {FACT}
        WHERE report_date >= DATE '2026-03-01'
          AND report_date < DATE '2026-05-01'
        GROUP BY 1, 2, 3
    ),
    paired AS (
        SELECT
            march.client_hash_id,
            march.content_hash_id,
            march.impressions AS impressions_march,
            march.clicks AS clicks_march,
            march.avg_position AS avg_position_march,
            march.sessions AS sessions_march,
            100.0 * march.engaged_sessions / NULLIF(march.sessions, 0) AS engagement_rate_march,
            april.impressions AS impressions_april,
            march.ga4_available
        FROM monthly AS march
        LEFT JOIN monthly AS april
          ON april.client_hash_id = march.client_hash_id
         AND april.content_hash_id = march.content_hash_id
         AND april.month_start = DATE '2026-04-01'
        WHERE march.month_start = DATE '2026-03-01'
    )
    SELECT *,
           CAST(impressions_april < 0.8 * impressions_march AS INTEGER)
               AS is_declining_next_month
    FROM paired
    WHERE ga4_available IS TRUE
      AND impressions_april IS NOT NULL
      AND impressions_march > 0
""").df()

feature_cols = [
    "impressions_march",
    "clicks_march",
    "avg_position_march",
    "sessions_march",
    "engagement_rate_march",
]
feature_frame = feature_frame.dropna(subset=feature_cols + ["is_declining_next_month"])
print(f"Feature rows with March GA4 availability and an April outcome: {len(feature_frame):,}")
print("Feature columns:", feature_cols)
display(feature_frame[feature_cols + ["is_declining_next_month"]].head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows with March GA4 availability and an April outcome: 71,482
Feature columns: ['impressions_march', 'clicks_march', 'avg_position_march', 'sessions_march', 'engagement_rate_march']


,impressions_march,clicks_march,avg_position_march,sessions_march,engagement_rate_march,is_declining_next_month
0,248.0,0.0,25.810492,5.0,0.000000,1
1,356.0,2.0,15.280320,7.0,14.285714,1
2,56.0,0.0,62.605882,1.0,0.000000,0
3,2289.0,3.0,5.789529,7.0,14.285714,1
4,1032.0,1.0,8.879862,6.0,0.000000,1


## 3b. Deliberate leakage experiment

*I add one label-derived column on purpose, measure the inflated result, then delete it before keeping the honest feature list.*

In [5]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

model_frame = feature_frame.copy()
model_frame["deliberate_label_leak"] = model_frame["is_declining_next_month"]
train_idx, test_idx = train_test_split(
    model_frame.index,
    test_size=0.25,
    random_state=42,
    stratify=model_frame["is_declining_next_month"],
)

y = model_frame["is_declining_next_month"]
leaky_auc = roc_auc_score(y.loc[test_idx], model_frame.loc[test_idx, "deliberate_label_leak"])

honest_model = make_pipeline(
    SimpleImputer(strategy="median"),
    LogisticRegression(max_iter=1000, random_state=42),
)
honest_model.fit(model_frame.loc[train_idx, feature_cols], y.loc[train_idx])
honest_auc = roc_auc_score(
    y.loc[test_idx],
    honest_model.predict_proba(model_frame.loc[test_idx, feature_cols])[:, 1],
)

print(f"Deliberate label-copy ROC-AUC: {leaky_auc:.3f} (invalid; expected 1.000)")
print(f"Honest March-only ROC-AUC: {honest_auc:.3f}")
print("Removing deliberate_label_leak before the retained feature frame.")
model_frame = model_frame.drop(columns=["deliberate_label_leak"])
assert "deliberate_label_leak" not in model_frame.columns
print("Retained model features:", feature_cols)

Deliberate label-copy ROC-AUC: 1.000 (invalid; expected 1.000)
Honest March-only ROC-AUC: 0.591
Removing deliberate_label_leak before the retained feature frame.
Retained model features: ['impressions_march', 'clicks_march', 'avg_position_march', 'sessions_march', 'engagement_rate_march']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice is observational and limited to clients with usable March GA4 availability and an observed April outcome. It cannot establish that any feature causes a decline, cannot describe pages without usable history, and cannot guarantee performance beyond this panel. Client history is unbalanced, GA4 availability is not universal, and the April label is a one-month operational proxy. The final June 2026 sample remains sealed and was not used for this experiment.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Named limitation: this is an observational, filtered March-to-April panel.")
print("It supports directional decision support, not causal claims or guaranteed future rankings.")
print("The sealed June 2026 sample was not used for label development.")

Named limitation: this is an observational, filtered March-to-April panel.
It supports directional decision support, not causal claims or guaranteed future rankings.
The sealed June 2026 sample was not used for label development.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.